In [0]:
from pyspark.sql.functions import *

# Read data quality table
dq_df = spark.table(
    "workspace.default.capstone_sales_quality"
)

# Keep only valid records
silver_df = (
    dq_df
    .filter(col("quality_flag") == "VALID")
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))
    .withColumn("city", trim(col("city")))
    .withColumn("state", trim(col("state")))
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("year", year(col("order_date")))
    .withColumn("month", month(col("order_date")))
    .withColumn("month_name", date_format(col("order_date"), "MMMM"))
)

# Save Silver table
silver_df.write.mode("overwrite").saveAsTable(
    "workspace.default.capstone_silver_sales"
)

print("Silver table created successfully")

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.default.capstone_silver_sales;

In [0]:
display(
    spark.sql("""
    SELECT order_date,
           year,
           month,
           month_name
    FROM workspace.default.capstone_silver_sales
    LIMIT 10
    """)
)

In [0]:
spark.sql("""
SELECT COUNT(*) AS total_records
FROM workspace.default.capstone_silver_sales
""").show()